# Why applications need databases

Why an application's data belongs in a database rather than files, what a relational database gives you (structure, rules, safe concurrent changes and fast lookups), and your first SQLite database from Python.

*From [CloudTech Academy](https://academy.cloudtechanalytics.com/learn/databases-and-apis-for-developers/why-applications-need-databases). Run the cells in order (Runtime → Run all), edit them, and experiment. Answer the practice questions on the lesson page.*

## The problem

Tallybook's invoicing service from the Software Engineering course still reads CSV exports. That works for a month-end report. It doesn't work for an app: two staff recording payments at the same moment can overwrite each other's changes, nothing stops a duplicate invoice or a discount of 50%, and finding one customer's invoices means reading every row in the file.

Applications keep their data in a **database**. In this course you'll design Tallybook's database, protect it with rules, change it safely, and put a real API in front of it, with every query and test run in Colab.

## The concept

### What a relational database provides

| Need | How the database helps |
| :-- | :-- |
| Structure | **tables** with named, typed columns |
| Relationships | **keys** link rows: each invoice belongs to a customer |
| Rules | **constraints** reject bad data at the door |
| Safe changes | **transactions** make a group of changes all happen or none |
| Speed | **indexes** find rows without reading the whole table |
| One question language | **SQL** |

![A Python app sends SQL and values to the database and gets rows back; inside the database are tables, constraints, transactions and indexes](https://academy.cloudtechanalytics.com/images/courses/dbapi/app-database.svg "The app sends SQL and values; the database stores, guards and finds the data.")

### SQLite

A complete relational database in a single file (or in memory), built into Python as `sqlite3`. Production systems often use PostgreSQL or MySQL; the ideas and almost all the SQL in this course carry over directly.

### From Python

In [ ]:
# This example shows a mistake, or needs something only you can provide.
# Run it to see what happens.
import sqlite3
conn = sqlite3.connect("tallybook.db")     # or ":memory:" for a throwaway database
conn.execute("SELECT ...", (value,))
conn.commit()

## Example

Create an in-memory database and load Tallybook's customers into a proper table:

In [ ]:
import sqlite3

import pandas as pd

base = "https://academy.cloudtechanalytics.com/datasets/invoicing/"
customers = pd.read_csv(base + "customers.csv")

conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE customers (
        customer_id        TEXT PRIMARY KEY,
        business_name      TEXT NOT NULL,
        city               TEXT NOT NULL,
        vat_exempt         INTEGER NOT NULL,
        payment_terms_days INTEGER NOT NULL
    )
""")
conn.executemany("INSERT INTO customers VALUES (?, ?, ?, ?, ?)", customers.itertuples(index=False))
conn.commit()

print(conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0], "customers")
for row in conn.execute("SELECT city, COUNT(*) AS n FROM customers GROUP BY city ORDER BY n DESC LIMIT 3"):
    print(row)

*Expected output:*

```
300 customers
('Lagos', 105)
('Kano', 52)
('Ibadan', 40)
```

Now try to add a customer whose ID already exists. A CSV file would accept it silently; the database refuses:

In [ ]:
try:
    conn.execute("INSERT INTO customers VALUES ('C0001', 'Duplicate Stores', 'Lagos', 0, 30)")
except sqlite3.IntegrityError as error:
    print("Refused:", error)

*Expected output:*

```
Refused: UNIQUE constraint failed: customers.customer_id
```

That refusal is the database doing a job your application code would otherwise have to remember to do, everywhere, forever.

## Walkthrough

1. Run the cells in Colab. Query the number of VAT-exempt customers.
2. Change `":memory:"` to `"tallybook.db"`, run again, and look for the file in Colab's file panel.
3. Why is `customer_id` the primary key rather than `business_name`?
4. List three things Tallybook's app needs that a CSV file can't provide safely.

## Practice


**Dataset:** [customers.csv](https://academy.cloudtechanalytics.com/datasets/invoicing/customers.csv), [invoices_raw.csv](https://academy.cloudtechanalytics.com/datasets/invoicing/invoices_raw.csv), [invoice_lines.csv](https://academy.cloudtechanalytics.com/datasets/invoicing/invoice_lines.csv)

**Practice:** How many customers are in **Lagos**?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Check your understanding